In [2]:
import pandas as pd
import numpy as np
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler

# 1. Load the processed feature matrix
input_path = '../data/processed/feature_matrix.csv'
if not os.path.exists(input_path):
    raise FileNotFoundError(f"Feature matrix not found at {input_path}. Run notebook 04 first.")

df = pd.read_csv(input_path)
feature_cols = [
    'absolute_magnitude_g',
    'bp_rp',
    'proper_motion_total',
    'tangential_velocity',
    'parallax_snr'
]
X = df[feature_cols].values

# 2. Standardize features for neural network stability
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Convert to PyTorch tensors
X_tensor = torch.tensor(X_scaled, dtype=torch.float32)
dataset = TensorDataset(X_tensor, X_tensor)
dataloader = DataLoader(dataset, batch_size=256, shuffle=True)

# 3. Define the Autoencoder Architecture
class AstronomicalAutoencoder(nn.Module):
    def __init__(self, input_dim):
        super(AstronomicalAutoencoder, self).__init__()
        # Encoder: Compress 5D -> 16D -> 8D -> 2D (Latent space)
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 16),
            nn.ReLU(),
            nn.Linear(16, 8),
            nn.ReLU(),
            nn.Linear(8, 2)
        )
        # Decoder: Reconstruct 2D -> 8D -> 16D -> 5D
        self.decoder = nn.Sequential(
            nn.Linear(2, 8),
            nn.ReLU(),
            nn.Linear(8, 16),
            nn.ReLU(),
            nn.Linear(16, input_dim)
        )

    def forward(self, x):
        latent = self.encoder(x)
        reconstructed = self.decoder(latent)
        return reconstructed

# 4. Initialize model, loss function, and optimizer
input_dim = X.shape[1]
model = AstronomicalAutoencoder(input_dim)
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.005)

# 5. Train the Autoencoder
epochs = 30
print(f"Training Autoencoder for {epochs} epochs...")
model.train()
for epoch in range(epochs):
    epoch_loss = 0.0
    for batch_x, _ in dataloader:
        optimizer.zero_grad()
        outputs = model(batch_x)
        loss = criterion(outputs, batch_x)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item() * batch_x.size(0)
    
    if (epoch + 1) % 5 == 0:
        print(f"Epoch [{epoch+1}/{epochs}], Loss: {epoch_loss / len(X_tensor):.4f}")

# 6. Calculate Reconstruction Error (Anomaly Score)
model.eval()
with torch.no_grad():
    reconstructions = model(X_tensor)
    # Calculate MSE per astronomical object
    mse = torch.mean((X_tensor - reconstructions) ** 2, dim=1).numpy()

# 7. Normalize scores to [0, 1] and rank
def min_max_scale(scores):
    return (scores - np.min(scores)) / (np.max(scores) - np.min(scores))

df['score_autoencoder'] = min_max_scale(mse)
df_ranked = df.sort_values(by='score_autoencoder', ascending=False).reset_index(drop=True)

# 8. Save Autoencoder results
output_dir = '../results/candidates'
os.makedirs(output_dir, exist_ok=True)
output_path = os.path.join(output_dir, 'ae_anomaly_candidates.csv')
df_ranked.to_csv(output_path, index=False)

print(f"\nAutoencoder scoring complete. Ranked candidates saved to {output_path}")
print("\nTop 10 Autoencoder Anomaly Candidates:")
display(df_ranked[['source_id', 'score_autoencoder'] + feature_cols].head(10))

Training Autoencoder for 30 epochs...
Epoch [5/30], Loss: 0.1075
Epoch [10/30], Loss: 0.0937
Epoch [15/30], Loss: 0.0836
Epoch [20/30], Loss: 0.0783
Epoch [25/30], Loss: 0.0763
Epoch [30/30], Loss: 0.0758

Autoencoder scoring complete. Ranked candidates saved to ../results/candidates/ae_anomaly_candidates.csv

Top 10 Autoencoder Anomaly Candidates:


,source_id,score_autoencoder,absolute_magnitude_g,bp_rp,proper_motion_total,tangential_velocity,parallax_snr
0,141972617872963968,1.000000,12.160490,3.096270,136.207978,45.888971,206.634815
1,141545595042005376,0.759316,7.183816,1.786256,194.486740,69.557588,58.090827
2,138831454885582336,0.678337,-1.982372,4.945676,3.807367,16.757030,13.291227
3,143293685388561664,0.516168,1.201925,0.504388,223.984597,41.079778,115.204385
4,139326028960063488,0.491594,11.704697,3.007754,67.720338,26.926087,190.187236
5,140403000598619136,0.374952,13.213551,4.027759,126.469641,18.773383,403.007150
6,139550531195676032,0.340453,7.644019,1.842097,241.629010,93.139634,126.933038
7,139134267259947008,0.322976,9.906199,2.473088,79.933926,59.250460,151.239256
8,143336257103994752,0.322357,6.056895,0.936204,164.755586,290.497213,145.263875
9,139257717503546368,0.321277,9.350436,2.532503,228.345225,37.186864,186.104967
